# 📊 Expenditure Analysis Dashboard

Analysis of categorized bank transactions from `data/expenditure_raw_with_category.csv`.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# Load data
DATA_PATH = Path("../data/expenditure_raw_with_category.csv")
df = pd.read_csv(DATA_PATH, sep=";", encoding="latin-1")

# Parse amounts: Debit is negative spending, Credit is positive income
def parse_amount(val):
    if pd.isna(val) or str(val).strip() == "":
        return 0.0
    s = str(val).replace("\xa0", "").replace(" ", "").replace("\u202f", "")
    if "," in s and "." not in s:
        s = s.replace(",", ".")
    else:
        s = s.replace(",", "")
    try:
        return float(s)
    except ValueError:
        return 0.0

df["debit"] = df["Debit"].apply(parse_amount).abs()  # positive number = money spent
df["credit"] = df["Credit"].apply(parse_amount)
df["amount"] = df["credit"] - df["debit"]  # negative = expense, positive = income
df["date"] = pd.to_datetime(df["Date de comptabilisation"], format="%d/%m/%Y", dayfirst=True)

# Filter out March (incomplete month)
march_rows = df[df["date"].dt.month == 3]
print(f"Removing {len(march_rows)} March rows (expenses: {march_rows['debit'].sum():,.2f} €, income: {march_rows['credit'].sum():,.2f} €)")
df = df[df["date"].dt.month != 3].copy()

# Filter out September (incomplete month)
september_rows = df[df["date"].dt.month == 9]
print(f"Removing {len(september_rows)} September rows (expenses: {september_rows['debit'].sum():,.2f} €, income: {september_rows['credit'].sum():,.2f} €)")
df = df[df["date"].dt.month != 9].copy()

df["month"] = df["date"].dt.to_period("M")
df["month_str"] = df["date"].dt.strftime("%Y-%m")
df["category"] = df["predicted_category"]

# Separate expenses and income
expenses = df[df["amount"] < 0].copy()
expenses["expense"] = expenses["amount"].abs()
income = df[df["amount"] > 0].copy()

print(f"Total rows: {len(df)}")
print(f"Date range: {df['date'].min().strftime('%d/%m/%Y')} → {df['date'].max().strftime('%d/%m/%Y')}")
print(f"Total expenses: {expenses['expense'].sum():,.2f} €")
print(f"Total income:   {income['amount'].sum():,.2f} €")
print(f"Net:            {df['amount'].sum():,.2f} €")

Removing 23 March rows (expenses: 223.47 €, income: 590.00 €)
Total rows: 500
Date range: 03/09/2025 → 28/02/2026
Total expenses: 10,177.05 €
Total income:   10,363.38 €
Net:            186.33 €


## 1. Spending by Category — Overview

In [24]:
# Total spending per category — pie chart + bar chart side by side
cat_totals = expenses.groupby("category")["expense"].sum().sort_values(ascending=False).reset_index()
cat_totals["pct"] = (cat_totals["expense"] / cat_totals["expense"].sum() * 100).round(1)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "bar"}]],
    subplot_titles=("Share of Total Spending", "Total Spent per Category (€)")
)

fig.add_trace(
    go.Pie(
        labels=cat_totals["category"],
        values=cat_totals["expense"],
        textinfo="label+percent",
        hovertemplate="%{label}<br>%{value:,.2f} €<br>%{percent}<extra></extra>",
        hole=0.35,
    ),
    row=1, col=1,
)

colors = px.colors.qualitative.Set2
fig.add_trace(
    go.Bar(
        x=cat_totals["category"],
        y=cat_totals["expense"],
        text=cat_totals["expense"].apply(lambda x: f"{x:,.0f} €"),
        textposition="outside",
        marker_color=colors[: len(cat_totals)],
        hovertemplate="%{x}<br>%{y:,.2f} €<extra></extra>",
    ),
    row=1, col=2,
)

fig.update_layout(
    title_text="Spending by Category",
    height=500,
    showlegend=False,
)
fig.update_yaxes(title_text="Amount (€)", row=1, col=2)
fig.show()

## 2. Monthly Spending Trend

In [25]:
# Monthly total expenses vs income
monthly = df.groupby("month_str").agg(
    total_expenses=("debit", "sum"),
    total_income=("credit", "sum"),
).reset_index().sort_values("month_str")

monthly["net"] = monthly["total_income"] - monthly["total_expenses"]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=monthly["month_str"], y=monthly["total_expenses"],
    name="Expenses", marker_color="#EF553B",
    hovertemplate="%{x}<br>Expenses: %{y:,.2f} €<extra></extra>",
))
fig.add_trace(go.Bar(
    x=monthly["month_str"], y=monthly["total_income"],
    name="Income", marker_color="#00CC96",
    hovertemplate="%{x}<br>Income: %{y:,.2f} €<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=monthly["month_str"], y=monthly["net"],
    name="Net", mode="lines+markers", line=dict(color="#636EFA", width=3),
    hovertemplate="%{x}<br>Net: %{y:,.2f} €<extra></extra>",
))

fig.update_layout(
    title="Monthly Income vs Expenses",
    xaxis_title="Month", yaxis_title="Amount (€)",
    barmode="group", height=450,
)
fig.show()

## 3. Monthly Spending Breakdown by Category

In [26]:
# Stacked bar: monthly expenses broken down by category
monthly_cat = expenses.groupby(["month_str", "category"])["expense"].sum().reset_index()
monthly_cat = monthly_cat.sort_values("month_str")

fig = px.bar(
    monthly_cat,
    x="month_str", y="expense", color="category",
    title="Monthly Spending by Category",
    labels={"month_str": "Month", "expense": "Amount (€)", "category": "Category"},
    color_discrete_sequence=px.colors.qualitative.Set2,
    height=500,
)
fig.update_layout(barmode="stack", xaxis_title="Month", yaxis_title="Amount (€)")
fig.show()

## 4. Top 20 Individual Expenses

In [27]:
# Top 20 biggest individual expenses
top20 = expenses.nlargest(20, "expense")[["date", "Libelle simplifie", "expense", "category"]].copy()
top20["label"] = top20["date"].dt.strftime("%d/%m") + " — " + top20["Libelle simplifie"]

fig = px.bar(
    top20.sort_values("expense"),
    x="expense", y="label", color="category",
    orientation="h",
    title="Top 20 Largest Individual Expenses",
    labels={"expense": "Amount (€)", "label": "", "category": "Category"},
    color_discrete_sequence=px.colors.qualitative.Set2,
    text="expense",
    height=600,
)
fig.update_traces(texttemplate="%{text:,.2f} €", textposition="outside")
fig.update_layout(yaxis=dict(categoryorder="total ascending"), showlegend=True)
fig.show()

## 5. Top Merchants by Total Spend

In [28]:
# Top 15 merchants/payees by cumulative spend
merchant_totals = (
    expenses.groupby("Libelle simplifie")
    .agg(total=("expense", "sum"), count=("expense", "count"), category=("category", "first"))
    .sort_values("total", ascending=False)
    .head(15)
    .reset_index()
)

fig = px.bar(
    merchant_totals.sort_values("total"),
    x="total", y="Libelle simplifie", color="category",
    orientation="h",
    title="Top 15 Merchants / Payees by Total Spend",
    labels={"total": "Total Spent (€)", "Libelle simplifie": "", "category": "Category"},
    color_discrete_sequence=px.colors.qualitative.Set2,
    text="total",
    hover_data={"count": True},
    height=550,
)
fig.update_traces(texttemplate="%{text:,.2f} €", textposition="outside")
fig.update_layout(yaxis=dict(categoryorder="total ascending"))
fig.show()

## 6. Daily Spending Timeline

In [29]:
# Daily spending with 7-day rolling average
daily = expenses.groupby("date")["expense"].sum().reset_index().sort_values("date")
daily["rolling_7d"] = daily["expense"].rolling(7, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=daily["date"], y=daily["expense"],
    name="Daily Spend", marker_color="lightcoral", opacity=0.6,
    hovertemplate="%{x|%d/%m/%Y}<br>%{y:,.2f} €<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=daily["date"], y=daily["rolling_7d"],
    name="7-day Average", line=dict(color="#EF553B", width=2.5),
    hovertemplate="%{x|%d/%m/%Y}<br>Avg: %{y:,.2f} €<extra></extra>",
))
fig.update_layout(
    title="Daily Spending with 7-Day Rolling Average",
    xaxis_title="Date", yaxis_title="Amount (€)",
    height=450,
)
fig.show()

## 7. Category Spending per Month — Heatmap

In [30]:
# Heatmap: category vs month
pivot = expenses.pivot_table(
    values="expense", index="category", columns="month_str", aggfunc="sum", fill_value=0
)
# Sort by total spend descending
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]
# Sort columns chronologically
pivot = pivot[sorted(pivot.columns)]

fig = px.imshow(
    pivot,
    text_auto=".0f",
    color_continuous_scale="OrRd",
    title="Spending Heatmap: Category × Month (€)",
    labels={"x": "Month", "y": "Category", "color": "€"},
    aspect="auto",
    height=450,
)
fig.update_layout(xaxis_title="Month", yaxis_title="")
fig.show()

## 8. Average Transaction Size per Category

In [31]:
# Box plot: distribution of transaction amounts per category
fig = px.box(
    expenses,
    x="category", y="expense", color="category",
    title="Transaction Amount Distribution per Category",
    labels={"expense": "Amount (€)", "category": "Category"},
    color_discrete_sequence=px.colors.qualitative.Set2,
    height=500,
    points="outliers",
)
fig.update_layout(showlegend=False, xaxis_title="", xaxis_tickangle=-30)
fig.show()

## 9. Cumulative Spending Over Time

In [32]:
# Cumulative spending per category over time
expenses_sorted = expenses.sort_values("date")

cat_cumsum = (
    expenses_sorted.groupby(["date", "category"])["expense"]
    .sum()
    .reset_index()
    .sort_values("date")
)
cat_cumsum["cumulative"] = cat_cumsum.groupby("category")["expense"].cumsum()

fig = px.area(
    cat_cumsum,
    x="date", y="cumulative", color="category",
    title="Cumulative Spending Over Time by Category",
    labels={"date": "Date", "cumulative": "Cumulative (€)", "category": "Category"},
    color_discrete_sequence=px.colors.qualitative.Set2,
    height=500,
)
fig.update_layout(xaxis_title="Date", yaxis_title="Cumulative Spent (€)")
fig.show()

## 10. Summary Statistics Table

In [33]:
# Summary stats per category
n_months = expenses["month_str"].nunique()

summary = (
    expenses.groupby("category")["expense"]
    .agg(["sum", "count", "mean", "median", "max"])
    .round(2)
    .sort_values("sum", ascending=False)
)
summary.columns = ["Total (€)", "# Transactions", "Avg (€)", "Median (€)", "Max (€)"]
summary["Monthly Avg (€)"] = (summary["Total (€)"] / n_months).round(2)
summary["% of Total"] = (summary["Total (€)"] / summary["Total (€)"].sum() * 100).round(1)

summary.style.format({
    "Total (€)": "{:,.2f}",
    "Avg (€)": "{:,.2f}",
    "Median (€)": "{:,.2f}",
    "Max (€)": "{:,.2f}",
    "Monthly Avg (€)": "{:,.2f}",
    "% of Total": "{:.1f}%",
})

,Total (€),# Transactions,Avg (€),Median (€),Max (€),Monthly Avg (€),% of Total
category,,,,,,,
Shopping,"2,305.21",81,28.46,15.20,279.99,384.20,22.7%
Finance & Transfers,"1,938.07",28,69.22,18.58,600.00,323.01,19.0%
Other,"1,411.99",66,21.39,5.00,628.49,235.33,13.9%
Food & Dining,"1,281.64",125,10.25,7.96,129.00,213.61,12.6%
Transport,"1,209.71",34,35.58,22.00,212.86,201.62,11.9%
Leisure & Culture,"1,184.95",72,16.46,10.00,150.00,197.49,11.6%
Education,355.00,3,118.33,5.00,348.00,59.17,3.5%
Housing & Utilities,254.07,14,18.15,3.99,142.00,42.34,2.5%
Health,236.41,21,11.26,5.90,42.92,39.40,2.3%
